# Урок 01 - Введение в AI Агентов

Добро пожаловать на первый урок курса **AI Агенты для начинающих**!

**AI агент** — это программа, которая использует большую языковую модель (LLM) в качестве движка рассуждений и может предпринимать *действия* в реальном мире — вызывать API, запрашивать базы данных или выполнять код — чтобы достичь цели от имени пользователя.

В этом блокноте вы создадите своего первого агента: **Туристический Агент**, который рекомендует места для отдыха. По ходу дела вы узнаете, как:

1. Подключиться к сервису Azure AI Foundry Agent с помощью **Microsoft Agent Framework**.
2. Дать агенту **инструмент** — обычную Python функцию, которую он может вызывать.
3. Запустить агента и проверить его ответ.
4. Передавать ответ агента по токену.


## Настройка

Прежде чем запускать эту тетрадь, убедитесь, что у вас есть:

1. **Проект Azure AI Foundry** с развернутой моделью чата (например, `gpt-4o-mini`).
2. **Вход в систему через Azure CLI** — выполните `az login` в вашем терминале.
3. **Установлены необходимые переменные окружения:**
   - `AZURE_AI_PROJECT_ENDPOINT` — конечная точка вашего проекта Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — имя вашей развернутой модели.

В ячейке ниже устанавливаются необходимые вам пакеты Python.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Создание вашего первого агента

Агенту нужно две вещи:

- **Инструкции**, которые говорят ему *кто он* и *как себя вести* (системный запрос).
- **Инструменты** — функции Python, декорированные с помощью `@tool`, которые агент может вызвать для получения информации или выполнения действий.

Ниже мы определяем простой инструмент, который возвращает список популярных мест для отдыха. Агент будет использовать этот инструмент, когда пользователь попросит рекомендации по путешествиям.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Потоковые ответы

Для более интерактивного взаимодействия вы можете **потоково** получать ответ агента. Вместо того чтобы ждать полного ответа, агент выдает текстовые фрагменты по мере их генерации. Это особенно полезно в чат-интерфейсах, где нужно отображать вывод в реальном времени.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Резюме

В этом уроке вы научились:

- **Создавать провайдера**, который подключается к Azure AI Foundry Agent Service через `AzureAIProjectAgentProvider`.
- **Определять инструмент** с помощью декоратора `@tool`, чтобы агент мог вызывать ваши функции Python.
- **Запускать агента** с пользовательским сообщением и выводить его ответ.
- **Потоковую передачу ответов** для вывода в реальном времени.

В следующем уроке мы более подробно изучим агентские фреймворки и научимся давать агентам более мощные инструменты и возможности многократного логического вывода.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:  
Этот документ был переведен с помощью сервиса автоматического перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия обеспечить точность, пожалуйста, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обращаться к профессиональному переводу, выполненному человеком. Мы не несем ответственности за любые недоразумения или неточные толкования, возникшие в результате использования данного перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
